# 1. GENERAR DATASET

In [81]:
import pandas as pd
import glob
import os

ruta = "PV_MaximumPowerPredictor/*.csv"

dfs = []

for archivo in glob.glob(ruta):
    
    nombre = os.path.basename(archivo).replace(".csv","")
    
    partes = nombre.split("_")
    
    parque_id = partes[0]              # Cocoa / Golden / Eugene
    panel_id = "_".join(partes[1:])    # tipo de panel
    
    df = pd.read_csv(archivo)
    
    # eliminar timestamp
    df = df.drop(columns=["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"], errors="ignore")
    
    # añadir columnas
    df["parque_id"] = parque_id
    df["panel_id"] = panel_id
    
    # mover columnas al inicio
    cols = ["parque_id","panel_id"] + [c for c in df.columns if c not in ["parque_id","panel_id"]]
    df = df[cols]
    
    dfs.append(df)

dataset_total = pd.concat(dfs, ignore_index=True)

print(dataset_total.shape)
dataset_total.head()

(1025599, 12)


,parque_id,panel_id,POA irradiance CMP22 pyranometer (W/m2),PV module back surface temperature (degC),Pmp (W),Dry bulb temperature (degC),Relative humidity (%RH),Atmospheric pressure (mb),Precipitation (mm) accumulated daily total,Direct normal irradiance (W/m2),Global horizontal irradiance (W/m2),Diffuse horizontal irradiance (W/m2)
0,Cocoa,aSiMicro03036_cleaned,20.2,19.3,1.9610,17.7,96.0,1009.8,20.3,0.1,21.9,22.2
1,Cocoa,aSiMicro03036_cleaned,35.8,19.5,3.7242,17.8,96.0,1009.7,20.3,0.1,38.4,39.0
2,Cocoa,aSiMicro03036_cleaned,20.2,19.5,1.9551,17.8,96.0,1009.9,20.3,0.1,20.1,20.3
3,Cocoa,aSiMicro03036_cleaned,20.6,19.5,2.0057,17.8,96.1,1009.8,20.3,0.1,21.4,21.8
4,Cocoa,aSiMicro03036_cleaned,29.5,19.5,3.1397,17.8,96.0,1009.9,20.6,0.1,29.3,29.5


# 2. EXPLORAR LOS DATOS

In [82]:
dataset_total["parque_id"].value_counts() # Comprobar num de parques y paneles

parque_id
Eugene    473348
Cocoa     421664
Golden    130587
Name: count, dtype: int64

In [83]:
dataset_total["panel_id"].nunique()

22

In [84]:
sorted(dataset_total["panel_id"].unique())

['CIGS1-001_cleaned',
 'CIGS39013_cleaned',
 'CIGS39017_cleaned',
 'CIGS8-001_cleaned',
 'CdTe75638_cleaned',
 'CdTe75669_cleaned',
 'HIT05662_cleaned',
 'HIT05667_cleaned',
 'aSiMicro03036_cleaned',
 'aSiMicro03038_cleaned',
 'aSiTandem72-46_cleaned',
 'aSiTandem90-31_cleaned',
 'aSiTriple28324_cleaned',
 'aSiTriple28325_cleaned',
 'mSi0166_cleaned',
 'mSi0188_cleaned',
 'mSi0247_cleaned',
 'mSi0251_cleaned',
 'mSi460A8_cleaned',
 'mSi460BB_cleaned',
 'xSi11246_cleaned',
 'xSi12922_cleaned']

In [85]:
pd.crosstab(dataset_total["parque_id"], dataset_total["panel_id"])

panel_id,CIGS1-001_cleaned,CIGS39013_cleaned,CIGS39017_cleaned,CIGS8-001_cleaned,CdTe75638_cleaned,CdTe75669_cleaned,HIT05662_cleaned,HIT05667_cleaned,aSiMicro03036_cleaned,aSiMicro03038_cleaned,...,aSiTriple28324_cleaned,aSiTriple28325_cleaned,mSi0166_cleaned,mSi0188_cleaned,mSi0247_cleaned,mSi0251_cleaned,mSi460A8_cleaned,mSi460BB_cleaned,xSi11246_cleaned,xSi12922_cleaned
parque_id,,,,,,,,,,,,,,,,,,,,,
Cocoa,0,0,34775,38939,39080,0,0,38377,39037,0,...,38485,0,36765,39102,0,0,38929,0,0,38989
Eugene,0,0,42674,43146,42248,0,0,43271,43343,0,...,42705,0,43268,43127,0,0,43115,0,0,43185
Golden,12011,11437,0,0,0,11953,11876,0,0,12148,...,0,11445,0,0,11912,11887,0,11919,11929,0


In [86]:
dataset_total.isna().sum()

parque_id                                     0
panel_id                                      0
POA irradiance CMP22 pyranometer (W/m2)       0
PV module back surface temperature (degC)     0
Pmp (W)                                       0
Dry bulb temperature (degC)                   0
Relative humidity (%RH)                       0
Atmospheric pressure (mb)                     0
Precipitation (mm) accumulated daily total    0
Direct normal irradiance (W/m2)               0
Global horizontal irradiance (W/m2)           0
Diffuse horizontal irradiance (W/m2)          0
dtype: int64

# 3. MODELO BASELINE

No ignoramos el parque y el panel, solo el timestamp.

In [87]:
from sklearn.preprocessing import LabelEncoder

le_parque = LabelEncoder()
le_panel  = LabelEncoder()

dataset_total["parque_enc"] = le_parque.fit_transform(dataset_total["parque_id"])
dataset_total["panel_enc"]  = le_panel.fit_transform(dataset_total["panel_id"])

In [88]:
df_model = dataset_total.drop(columns=["parque_id", "panel_id"])

In [89]:
X = df_model.drop(columns=["Pmp (W)"])
y = df_model["Pmp (W)"]

In [90]:
from sklearn.model_selection import train_test_split

# Primero separar test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Luego separar val del resto (20% del total = 25% de train_val)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

# Resultado: 60% train, 20% val, 20% test
print(X_train.shape, X_val.shape, X_test.shape)

(615359, 11) (205120, 11) (205120, 11)


In [91]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.fit_transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [92]:
import numpy as np

np.mean(X_train_scaled, axis=0)


array([-5.35540588e-17, -7.48036350e-16,  9.28362727e-18,  6.21795185e-17,
       -4.20880863e-18, -2.36132062e-17,  2.52153247e-17,  1.32037659e-17,
        4.07428841e-17,  8.67337889e-17,  3.26428039e-17])

In [93]:
np.std(X_train_scaled, axis=0)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [94]:
import torch

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

## A. DISEÑO DEL MODELO

In [95]:
import torch
import torch.nn as nn

class PVModel(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(11, 64),
            nn.ReLU(),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, 16),
            nn.ReLU(),
            
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.model(x)

In [96]:
class PVModelV2(nn.Module):
    
    def __init__(self, input_size=11):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.model(x)

In [97]:
model = PVModel()
print(model)

model2 = PVModelV2()
print(model2)

PVModel(
  (model): Sequential(
    (0): Linear(in_features=11, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)
PVModelV2(
  (model): Sequential(
    (0): Linear(in_features=11, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=32, out_features=16, bias=True)
    (9): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_feature

In [98]:
model(X_train_tensor[:5])

tensor([[0.3080],
        [0.3077],
        [0.3236],
        [0.2994],
        [0.3146]], grad_fn=<AddmmBackward0>)

In [99]:
model2(X_train_tensor[:5])

tensor([[ 0.1858],
        [ 0.4782],
        [-0.0651],
        [ 0.8272],
        [-0.1635]], grad_fn=<AddmmBackward0>)

In [100]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [101]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024
)

## B. ML FLOW

In [102]:
import mlflow

In [103]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("PV_Power")

<Experiment: artifact_location='file:///c:/Users/User/Documents/GitHub/Reto3_MicroRedes/OBJETIVO2/mlruns/427506226560148484', creation_time=1773741607353, experiment_id='427506226560148484', last_update_time=1773741607353, lifecycle_stage='active', name='PV_Power', tags={}, workspace='default'>

In [104]:
class PVModel(nn.Module):
    
    def __init__(self, layers_sizes, input_size=11):
        super().__init__()
        
        layers = []
        
        for size in layers_sizes:
            layers.append(nn.Linear(input_size, size))
            layers.append(nn.ReLU())
            input_size = size
        
        layers.append(nn.Linear(input_size, 1))
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.model(x)

In [105]:
class PVModelV2(nn.Module):
    
    def __init__(self, layers_sizes,dropout, input_size=11):
        super().__init__()
        
        layers = []
        
        for size in layers_sizes:
            layers.append(nn.Linear(input_size, size))
            layers.append(nn.BatchNorm1d(size))   # nuevo
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))         # nuevo
            input_size = size
        
        layers.append(nn.Linear(input_size, 1))
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.model(x)

In [106]:
def evaluate(model, loader, criterion):
    
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
    
    return total_loss / len(loader)

In [107]:
def train_model(arch, lr, epochs, model,dropout=0):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    patience = 5
    counter = 0
    
    with mlflow.start_run():
    
        mlflow.log_param("arquitectura", str(arch))
        mlflow.log_param("lr", lr)
        mlflow.log_param("dropout",dropout)
        for epoch in range(epochs):
            
            model.train()
            total_loss = 0
            
            for X_batch, y_batch in train_loader:
                
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            train_loss = total_loss / len(train_loader)
            val_loss = evaluate(model, val_loader, criterion)
            
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            
            print(f"Epoch {epoch+1} | Train: {train_loss:.2f} | Val: {val_loss:.2f}")

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), f"best_model.pt")
                counter = 0
            else:
                counter += 1
            if counter >= patience:
                print(f"Early stopping en epoch {epoch+1}")
                break

    return model

In [108]:
arquitecturas = [
    [64, 32, 16],
    [128, 64, 32],
    [256, 128, 64]
]

learning_rates = [0.001, 0.0005]

dropouts = [0.1, 0.2]

epochs = 100

In [110]:
for arch in arquitecturas:
    for lr in learning_rates:
        model = PVModel(arch)
        print(f"\nEntrenando modelo {arch} con lr={lr}")
        
        train_model(arch, lr, epochs,model)


Entrenando modelo [64, 32, 16] con lr=0.001
Epoch 1 | Train: 821.36 | Val: 480.43
Epoch 2 | Train: 329.67 | Val: 243.12
Epoch 3 | Train: 195.12 | Val: 145.56
Epoch 4 | Train: 110.22 | Val: 86.81
Epoch 5 | Train: 73.26 | Val: 65.06
Epoch 6 | Train: 58.54 | Val: 53.41
Epoch 7 | Train: 51.18 | Val: 48.83
Epoch 8 | Train: 46.79 | Val: 48.12
Epoch 9 | Train: 43.82 | Val: 43.69
Epoch 10 | Train: 42.20 | Val: 41.61
Epoch 11 | Train: 40.14 | Val: 38.70
Epoch 12 | Train: 38.55 | Val: 37.07
Epoch 13 | Train: 36.96 | Val: 37.19
Epoch 14 | Train: 35.62 | Val: 35.71
Epoch 15 | Train: 34.36 | Val: 37.25
Epoch 16 | Train: 32.92 | Val: 32.96
Epoch 17 | Train: 31.93 | Val: 31.61
Epoch 18 | Train: 30.92 | Val: 32.61
Epoch 19 | Train: 30.23 | Val: 29.70
Epoch 20 | Train: 29.47 | Val: 28.49
Epoch 21 | Train: 28.46 | Val: 28.22
Epoch 22 | Train: 27.71 | Val: 27.16
Epoch 23 | Train: 27.17 | Val: 25.96
Epoch 24 | Train: 26.25 | Val: 27.46
Epoch 25 | Train: 25.74 | Val: 25.34
Epoch 26 | Train: 25.11 | Val: 2

In [111]:
for arch in arquitecturas:
    for lr in learning_rates:
        for dropout in dropouts:   
            
            model = PVModelV2(arch, dropout=dropout)
            
            print(f"\nEntrenando modelo {arch} | lr={lr} | dropout={dropout}")
            
            train_model(arch, lr, epochs, model)


Entrenando modelo [64, 32, 16] | lr=0.001 | dropout=0.1
Epoch 1 | Train: 1749.05 | Val: 879.97
Epoch 2 | Train: 401.29 | Val: 108.48
Epoch 3 | Train: 155.66 | Val: 64.96
Epoch 4 | Train: 135.01 | Val: 61.31
Epoch 5 | Train: 121.99 | Val: 43.83
Epoch 6 | Train: 112.01 | Val: 40.40
Epoch 7 | Train: 104.86 | Val: 30.23
Epoch 8 | Train: 97.67 | Val: 29.40
Epoch 9 | Train: 93.84 | Val: 24.24
Epoch 10 | Train: 89.35 | Val: 24.73
Epoch 11 | Train: 86.85 | Val: 21.37
Epoch 12 | Train: 84.70 | Val: 17.78
Epoch 13 | Train: 81.29 | Val: 18.82
Epoch 14 | Train: 80.09 | Val: 20.07
Epoch 15 | Train: 78.90 | Val: 15.88
Epoch 16 | Train: 78.71 | Val: 16.36
Epoch 17 | Train: 77.22 | Val: 15.74
Epoch 18 | Train: 72.68 | Val: 17.06
Epoch 19 | Train: 74.87 | Val: 12.34
Epoch 20 | Train: 72.04 | Val: 12.87
Epoch 21 | Train: 71.34 | Val: 13.97
Epoch 22 | Train: 71.40 | Val: 13.85
Epoch 23 | Train: 69.24 | Val: 12.65
Epoch 24 | Train: 69.47 | Val: 13.40
Early stopping en epoch 24

Entrenando modelo [64, 32,

# MODELOS SIMPLES

In [112]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Entrenar modelo
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predicción
y_pred = lr_model.predict(X_val)

# Métricas
mse = mean_squared_error(y_val, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_val, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

MSE: 576.3045725256211
RMSE: 24.006344422373456
R2: 0.5934330172716631


In [113]:
y_pred_nn = model(X_val_tensor).detach().numpy()

In [114]:
mean_squared_error(y_val, y_pred_nn)

4.569985740959549

In [115]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Modelo
model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Entrenar
model.fit(X_train, y_train)

# Predecir
y_pred = model.predict(X_val)

# Métricas
mse = mean_squared_error(y_val, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_val, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

MSE: 2.57751123635138
RMSE: 1.6054629352156904
R2: 0.9981816369047373


# 4. FEATURE ENGENIERING

In [116]:
corr = df_model.corr(numeric_only=True)
corr["Pmp (W)"].sort_values(ascending=False)

Pmp (W)                                       1.000000
POA irradiance CMP22 pyranometer (W/m2)       0.748333
PV module back surface temperature (degC)     0.561800
Direct normal irradiance (W/m2)               0.068805
Global horizontal irradiance (W/m2)           0.057096
Precipitation (mm) accumulated daily total   -0.018312
Dry bulb temperature (degC)                  -0.019508
Diffuse horizontal irradiance (W/m2)         -0.023923
Atmospheric pressure (mb)                    -0.031497
Relative humidity (%RH)                      -0.033575
parque_enc                                   -0.036538
panel_enc                                    -0.179015
Name: Pmp (W), dtype: float64

In [117]:
df_model["physical_model"] = (
    df_model["POA irradiance CMP22 pyranometer (W/m2)"] *
    (1 - 0.004 * (df_model["PV module back surface temperature (degC)"] - 25))
)

In [118]:
df_model = df_model[[
    "POA irradiance CMP22 pyranometer (W/m2)",
    "PV module back surface temperature (degC)",
    "physical_model",
    "Pmp (W)"
]]

In [119]:
X = df_model.drop(columns=["Pmp (W)"])
y = df_model["Pmp (W)"]

In [120]:
from sklearn.model_selection import train_test_split

# Primero separar test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Luego separar val del resto (20% del total = 25% de train_val)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

# Resultado: 60% train, 20% val, 20% test
print(X_train.shape, X_val.shape, X_test.shape)

(615359, 3) (205120, 3) (205120, 3)


In [121]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.fit_transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [122]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024
)

In [123]:
for arch in arquitecturas:
    for lr in learning_rates:
        model = PVModel(arch)
        print(f"\nEntrenando modelo {arch} con lr={lr}")
        train_model(arch, lr, epochs,model)


Entrenando modelo [64, 32, 16] con lr=0.001
Epoch 1 | Train: 831.11 | Val: 469.28
Epoch 2 | Train: 356.33 | Val: 304.37
Epoch 3 | Train: 269.90 | Val: 237.96
Epoch 4 | Train: 192.68 | Val: 153.05
Epoch 5 | Train: 124.72 | Val: 104.05
Epoch 6 | Train: 88.79 | Val: 77.27
Epoch 7 | Train: 70.60 | Val: 64.29
Epoch 8 | Train: 62.22 | Val: 57.51
Epoch 9 | Train: 55.98 | Val: 53.70
Epoch 10 | Train: 51.60 | Val: 49.95
Epoch 11 | Train: 48.20 | Val: 45.22
Epoch 12 | Train: 45.24 | Val: 45.83
Epoch 13 | Train: 43.49 | Val: 41.66
Epoch 14 | Train: 41.55 | Val: 38.84
Epoch 15 | Train: 39.24 | Val: 38.53
Epoch 16 | Train: 36.37 | Val: 34.78
Epoch 17 | Train: 33.96 | Val: 34.09
Epoch 18 | Train: 32.36 | Val: 30.75
Epoch 19 | Train: 29.91 | Val: 27.83
Epoch 20 | Train: 27.84 | Val: 28.25
Epoch 21 | Train: 26.12 | Val: 24.92
Epoch 22 | Train: 24.58 | Val: 22.69
Epoch 23 | Train: 23.23 | Val: 21.92
Epoch 24 | Train: 21.14 | Val: 22.92
Epoch 25 | Train: 20.13 | Val: 21.81
Epoch 26 | Train: 19.35 | Val

In [124]:
for arch in arquitecturas:
    for lr in learning_rates:
        for dropout in dropouts:  
            
            model = PVModelV2(arch, dropout=dropout)
            
            print(f"\nEntrenando modelo {arch} | lr={lr} | dropout={dropout}")
            
            train_model(arch, lr, epochs, model)


Entrenando modelo [64, 32, 16] | lr=0.001 | dropout=0.1
Epoch 1 | Train: 2044.93 | Val: 1289.77
Epoch 2 | Train: 546.41 | Val: 111.36
Epoch 3 | Train: 145.20 | Val: 53.61
Epoch 4 | Train: 120.15 | Val: 40.98
Epoch 5 | Train: 106.37 | Val: 50.62
Epoch 6 | Train: 98.34 | Val: 27.53
Epoch 7 | Train: 93.51 | Val: 20.17
Epoch 8 | Train: 85.71 | Val: 24.15
Epoch 9 | Train: 82.17 | Val: 19.88
Epoch 10 | Train: 79.16 | Val: 19.27
Epoch 11 | Train: 74.24 | Val: 19.24
Epoch 12 | Train: 73.49 | Val: 20.91
Epoch 13 | Train: 70.00 | Val: 29.13
Epoch 14 | Train: 67.84 | Val: 19.68
Epoch 15 | Train: 66.18 | Val: 20.77
Epoch 16 | Train: 63.68 | Val: 19.87
Early stopping en epoch 16

Entrenando modelo [64, 32, 16] | lr=0.001 | dropout=0.2
Epoch 1 | Train: 1893.08 | Val: 1039.87
Epoch 2 | Train: 438.16 | Val: 145.83
Epoch 3 | Train: 183.81 | Val: 77.51
Epoch 4 | Train: 167.10 | Val: 59.93
Epoch 5 | Train: 156.16 | Val: 56.66
Epoch 6 | Train: 148.31 | Val: 43.32
Epoch 7 | Train: 141.11 | Val: 50.83
Epoc

# ENTRENAMIENTOS FINALES DE LOS MEJORES MODELOS